# Step 3: Dataset Formatting & LLM Fine-Tuning

This notebook demonstrates the process of building the dataset using a teacher LLM (Gemini) and then fine-tuning a small on-device LLM (like Qwen or SmolLM).

## 0a. Google Colab Setup (Step 1)
Run this cell FIRST to install Conda in Google Colab.
**Note:** Colab will automatically restart the kernel after this cell finishes. This is normal! Wait for it to reconnect before moving to Step 1b.

In [1]:
try:
    import google.colab
    !pip install -q condacolab
    import condacolab
    condacolab.install()
except ImportError:
    print("Not running in Colab. Skipping Conda setup.")

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:09
🔁 Restarting kernel...


## 0b. Mount Drive & Load Environment (Step 2)
After the kernel restarts, run this cell to mount your Google Drive, navigate to the project folder, and install all dependencies from `environment.yml`.

In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"

    print("\n--- Installing Environment ---")
    !conda env update -n base -f environment.yml

    print("\n--- Installing Colab Unsloth Drivers ---")
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps "xformers<0.0.27" peft accelerate bitsandbytes
except ImportError:
    print("Not running in Colab. Skipping mount and environment update.")

Mounted at /content/drive
/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor

--- Installing Environment ---
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: \ failed

SpecsConfigurationConflictError: Requested specs conflict with configured specs.
  requested specs: 
    - ffmpeg=8.0.1
    - pip
    - python=3.11.15
  pinned specs: 
    - cuda-version=12
    - python=3.12
    - python_abi=3.12[build=*cp312*]
Use 'conda config --show-sources' to look for 'pinned_specs' and 'track_features'
configuration parameters.  Pinned specs may also be defined in the file
/usr/local/conda-meta/pinned.



--- Installing Colab Unsloth Drivers ---
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-5ejn4rcd/unsloth_62741a0833ca4bdcbd1e8efb14483c21
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.gi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 MB 67.1 MB/s eta 0:00:00


## 1. Dataset Generation
Run the `dataset_builder.py` script. It will scan all `final_dialogue.json` files and use Gemini to extract structured CRM fields. It includes checkpointing, so you can stop and resume anytime.

In [ ]:
!python pipeline/dataset_builder.py

### Preview the generated dataset

In [1]:
import json
import pandas as pd

try:
    %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"
    with open('data/finetuning_dataset.jsonl', 'r', encoding='utf-8') as f:
        # Load first 5 records
        data = [json.loads(next(f)) for _ in range(5)]
    df = pd.DataFrame(data)
    display(df)
except Exception as e:
    print("Run the dataset builder first to generate the dataset.", e)

/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor


,instruction,input,response
0,You are a structured-field extractor for teles...,Agent: à dạ a lô ạ à chị ạ em là hòa trương vi...,"{""customer_sector"": ""Bất động sản"", ""customer_..."
1,You are a structured-field extractor for teles...,Agent: a lô cho em hỏi đây cái số chị của anh\...,"{""customer_sector"": null, ""customer_needs"": ""i..."
2,You are a structured-field extractor for teles...,Customer: a lô\nAgent: vẻ luôn ạ dạ em chào ch...,"{""customer_sector"": null, ""customer_needs"": ""T..."
3,You are a structured-field extractor for teles...,Customer: a lô\nAgent: dạ em chào anh ạ anh có...,"{""customer_sector"": ""Retail and Dental clinic""..."
4,You are a structured-field extractor for teles...,Customer: a lô\nAgent: a lô à dạ vâng em chào ...,"{""customer_sector"": ""Bất động sản"", ""customer_..."


## 2. Model Fine-Tuning
Now we fine-tune a small LLM using LoRA via the `unsloth` library. We are running consecutively over:
- `Qwen/Qwen3.5-2B`
- `Qwen/Qwen3.5-0.8B`
- `google/gemma-4-E4B`

In [ ]:
!python pipeline/fine_tuner.py --model Qwen/Qwen3.5-2B --save_name Qwen3.5-2B
# !python pipeline/fine_tuner.py --model Qwen/Qwen3.5-0.8B --save_name Qwen3.5-0.8B
# !python pipeline/fine_tuner.py --model google/gemma-4-E4B --save_name gemma-4-E4B

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W0522 10:06:07.831000 3440 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0522 10:06:07.878000 3440 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
🦥 Unsloth Zoo will now patch everything to make training faster!
<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
Loading model: Qwen/Qwen3.5-2B
==((====))==  Unsloth 2026.5.5: Fast Qwe

## 3. Inference / Evaluation
Once the model is fine-tuned, you can load the generated LoRA weights to run inference on new transcripts.

In [ ]:
!python pipeline/inference.py --model data/models/Qwen3.5-2B_lora

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W0521 15:38:44.278000 21109 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0521 15:38:44.320000 21109 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
🦥 Unsloth Zoo will now patch everything to make training faster!
<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
Loading finetuned model from: data/models/Qwen3.5-2B_lora...
==((====